## File 7 — `07_Functools_Wraps.ipynb`

This file should focus specifically on **`functools.wraps`** and why it is important when writing decorators.

We should **not** reteach decorators, closures, `*args`, `**kwargs`, or decorator arguments. Those have already been covered.

### Structure

```text
07_Functools_Wraps.ipynb

1. Introduction

2. The Problem with Decorators and Function Metadata

3. What Happens to the Original Function?

4. Function Metadata
   - __name__
   - __doc__
   - __module__
   - __qualname__
   - __annotations__

5. Demonstrating Metadata Loss

6. Why Metadata Matters

7. Introduction to functools

8. Importing functools

9. What Is functools.wraps?

10. Using @wraps

11. How @wraps Works

12. Before and After @wraps

13. Preserving Function Documentation

14. Preserving Function Name

15. Preserving Other Function Metadata

16. Using @wraps with *args and **kwargs
    (Reference Only)

17. Practical Decorator Examples

18. Common Mistakes

19. Summary
```

---

# 1. Introduction

Start with a decorator we've already learned:

In [ ]:
def decorator(func):
    def wrapper():
        print("Before")
        return func()
    return wrapper

Then:

In [ ]:
@decorator
def greet():
    """Say hello to the user."""
    print("Hello")

The decorator works correctly.

But inspect:

In [ ]:
print(greet.__name__)
print(greet.__doc__)

You may get:

```text
wrapper
None
```

instead of:

```text
greet
Say hello to the user.
```

This is the problem this notebook solves.

---

# 2. The Problem with Decorators and Function Metadata

Explain that:

In [ ]:
@decorator
def greet():
    ...

causes `greet` to refer to the **wrapper function** after decoration.

Conceptually:

In [ ]:
greet = decorator(greet)

Therefore:

In [ ]:
greet

now points to `wrapper`.

This means information about the original function can be hidden or replaced.

---

# 3. What Happens to the Original Function?

Demonstrate:

In [ ]:
def decorator(func):

    def wrapper():
        return func()

    return wrapper

Then:

In [ ]:
@decorator
def greet():
    """Say hello."""
    print("Hello")

Check:

In [ ]:
print(greet)
print(greet.__name__)
print(greet.__doc__)

Explain that the decorated function now exposes the wrapper's metadata.

---

# 4. Function Metadata

Introduce the metadata that matters for this lesson.

### `__name__`

In [ ]:
print(greet.__name__)

### `__doc__`

In [ ]:
print(greet.__doc__)

### `__module__`

In [ ]:
print(greet.__module__)

### `__qualname__`

In [ ]:
print(greet.__qualname__)

### `__annotations__`

In [ ]:
print(greet.__annotations__)

The goal is not to turn this into a separate function-metadata lesson. Only explain these attributes enough to understand what `wraps` preserves.

---

# 5. Demonstrating Metadata Loss

Use a complete example:

In [ ]:
def decorator(func):

    def wrapper():
        """Wrapper documentation."""
        return func()

    return wrapper


@decorator
def greet():
    """Original greeting function."""
    print("Hello")

Then:

In [ ]:
print(greet.__name__)
print(greet.__doc__)

Expected:

```text
wrapper
Wrapper documentation.
```

Explain why.

---

# 6. Why Metadata Matters

Explain practical reasons:

- debugging
- documentation
- introspection
- development tools
- testing
- logging
- help systems
- understanding decorated functions

For example:

In [ ]:
help(greet)

A decorator shouldn't unnecessarily make a function appear to be something completely different.

---

# 7. Introduction to `functools`

Introduce Python's standard-library module:

In [ ]:
import functools

Explain that `functools` provides utilities for working with functions.

For this notebook, the important tool is:

In [ ]:
functools.wraps

Don't turn this into a broad `functools` module tutorial.

---

# 8. Importing `functools`

Show both common styles:

In [ ]:
import functools

and:

In [ ]:
from functools import wraps

Then explain that the second form allows:

In [ ]:
@wraps(func)

instead of:

In [ ]:
@functools.wraps(func)

---

# 9. What Is `functools.wraps`?

Core explanation:

> `functools.wraps` is a helper used inside decorators to preserve important metadata from the original function on the wrapper.

Without it:

In [ ]:
def decorator(func):

    def wrapper():
        return func()

    return wrapper

With it:

In [ ]:
from functools import wraps

def decorator(func):

    @wraps(func)
    def wrapper():
        return func()

    return wrapper

---

# 10. Using `@wraps`

This is the central example:

In [ ]:
from functools import wraps


def decorator(func):

    @wraps(func)
    def wrapper():
        print("Before")
        result = func()
        print("After")
        return result

    return wrapper

Then:

In [ ]:
@decorator
def greet():
    """Say hello to the user."""
    print("Hello")

Now:

In [ ]:
print(greet.__name__)
print(greet.__doc__)

gives the original function's metadata.

---

# 11. How `@wraps` Works

Explain the syntax:

In [ ]:
@wraps(func)
def wrapper():
    ...

Conceptually, `wraps(func)` configures the wrapper so that important information from `func` is copied onto it.

Keep this conceptual rather than diving into the implementation of `update_wrapper`.

---

# 12. Before and After `@wraps`

This section should provide a direct comparison.

### Without `@wraps`

In [ ]:
def decorator(func):

    def wrapper():
        return func()

    return wrapper

Result:

```text
__name__ → wrapper
__doc__  → wrapper's metadata
```

### With `@wraps`

In [ ]:
from functools import wraps

def decorator(func):

    @wraps(func)
    def wrapper():
        return func()

    return wrapper

Result:

```text
__name__ → original function name
__doc__  → original function documentation
```

This comparison is one of the most important parts of the notebook.

---

# 13. Preserving Function Documentation

Example:

In [ ]:
from functools import wraps


def log_call(func):

    @wraps(func)
    def wrapper():
        print("Calling function...")
        return func()

    return wrapper


@log_call
def calculate():
    """Perform a calculation."""
    print("Calculating...")

Then:

In [ ]:
print(calculate.__name__)
print(calculate.__doc__)

Expected:

```text
calculate
Perform a calculation.
```

---

# 14. Preserving Function Name

Show:

In [ ]:
print(calculate.__name__)

Without `@wraps`:

```text
wrapper
```

With `@wraps`:

```text
calculate
```

This makes the purpose of `wraps` very concrete.

---

# 15. Preserving Other Function Metadata

Demonstrate:

In [ ]:
def greet(name: str) -> str:
    """Return a greeting."""
    return f"Hello {name}"

Then inspect:

In [ ]:
print(greet.__annotations__)

After decoration with `@wraps`, explain that important metadata such as annotations is preserved/copied as part of the wrapper metadata handling.

Don't go deeply into annotations themselves because that's outside the purpose of this notebook.

---

# 16. Using `@wraps` with `*args` and `**kwargs`

**Reference only.**

We already covered `*args` and `**kwargs` in:

```text
01_Functions_and_Scope/
└── 03_Args_and_Kwargs.ipynb
```

Here they are only used to make the decorator flexible:

In [ ]:
from functools import wraps


def log_call(func):

    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        result = func(*args, **kwargs)
        return result

    return wrapper

Example:

In [ ]:
@log_call
def add(a, b):
    """Add two numbers."""
    return a + b

Then:

In [ ]:
print(add(10, 20))
print(add.__name__)
print(add.__doc__)

The notebook should say:

> `*args` and `**kwargs` are used here but are not explained again. See `01_Functions_and_Scope/03_Args_and_Kwargs.ipynb`.

---

# 17. Practical Decorator Examples

### Example 1 — Logging

In [ ]:
from functools import wraps


def log_function(func):

    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} finished")
        return result

    return wrapper

### Example 2 — Timing-style decorator

Don't need to teach timing deeply; just demonstrate the pattern.

In [ ]:
from functools import wraps
import time


def measure_time(func):

    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()

        result = func(*args, **kwargs)

        end = time.time()

        print("Execution time:", end - start)

        return result

    return wrapper

### Example 3 — Validation-style decorator

In [ ]:
from functools import wraps


def validate(func):

    @wraps(func)
    def wrapper(*args, **kwargs):
        print("Validating...")
        return func(*args, **kwargs)

    return wrapper

The focus remains on **why** **`@wraps`** **belongs inside these decorators**.

---

# 18. Common Mistakes

### Mistake 1 — Forgetting `@wraps`

In [ ]:
def decorator(func):

    def wrapper():
        return func()

    return wrapper

Explain the metadata problem.

---

### Mistake 2 — Putting `@wraps` in the wrong place

Correct:

In [ ]:
def decorator(func):

    @wraps(func)
    def wrapper():
        return func()

    return wrapper

Not:

In [ ]:
@wraps
def decorator(func):
    ...

---

### Mistake 3 — Using `wraps` without passing the original function

Correct:

In [ ]:
@wraps(func)

because `func` is the function whose metadata we want to preserve.

---

### Mistake 4 — Thinking `wraps` changes the decorator's behavior

Clarify:

`@wraps` **does not perform the decoration itself**.

It helps the wrapper preserve the original function's metadata.

---

# 19. Summary

End with this mental model:

```text
Original Function
       │
       │ passed to decorator
       ▼
    decorator
       │
       ▼
    wrapper
       │
       │ @wraps(func)
       ▼
Wrapper keeps important
metadata from original function
```

### Key takeaway

Without:

In [ ]:
@wraps(func)

a decorated function can look like:

```text
wrapper
```

With:

In [ ]:
@wraps(func)

it continues to look like the original function:

```text
original function name
original documentation
important metadata
```

### What this notebook should NOT cover

```text
✗ Basic decorators again
✗ Decorator arguments again
✗ *args / **kwargs tutorial
✗ Closures tutorial
✗ Function scope
✗ Class decorators
✗ Advanced functools utilities
```

Those belong elsewhere in the curriculum.

**Next file:** `08_Decorators_with_Function_Arguments.ipynb` — where we'll focus on decorators that correctly handle functions receiving their own arguments and return values.